# Reporte de Proyecto: Diccionario y Ordenamiento

En este reporte voy a explicar paso a paso como resolvimos el ejercicio de crear un diccionario a partir de un texto. El objetivo principal es no usar herramientas integradas de Python como `set` o `sort()` (POO).

Para que sea facil de entender, dividimos el programa en 4 clases (como si fueran departamentos de una empresa). Cada clase hace una sola cosa especifica. ¡Vamos a verlas una por una!

## 1. Clase Normalizador

Esta es la primera parte de mi programa. Su trabajo es recibir el texto sucio y entregarlo limpio. 
Lo que hacemos aqui es:
1. Validar que el texto no este vacio (si lo esta, lanzo un error).
2. Convertir todas las letras a minusculas.
3. Recorrer letra por letra para cambiar las que tienen acento por letras normales.
4. Quitar los numeros y signos de puntuacion, cambiandolos por espacios en blanco.
5. Al final, usamos `split()` para separar todo por los espacios y devolver una lista de palabras.

In [ ]:
import time

class Normalizador:
    def __init__(self):
        # Diccionario simple para quitar tildes a las vocales
        self.mapa_letras = {
            'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
            'Á': 'a', 'É': 'e', 'Í': 'i', 'Ó': 'o', 'Ú': 'u'
        }

    def limpiar_texto(self, texto):
        # Si el texto no tiene nada, avisamos del error
        if texto == "" or texto.isspace():
            raise ValueError("Error: El texto esta vacio.")

        texto_limpio = ""
        texto_minusculas = texto.lower()

        for letra in texto_minusculas:
            if letra in self.mapa_letras:
                texto_limpio += self.mapa_letras[letra]
            elif letra.isalpha() or letra.isspace():
                # Conservamos letras del abecedario y espacios
                texto_limpio += letra
            else:
                # Cualquier otra cosa (numeros, comas) se vuelve espacio
                texto_limpio += " "

        # Regresamos la lista de palabras
        return texto_limpio.split()


## 2. Clase TablaHash (Nuestro Archivero)

Como no nos dejan usar `set` para eliminar repetidos, hicimos nuestra propia Tabla Hash.
Imaginemos que creo 100 cubetas (o buckets) vacias.

Cuando me llega una palabra, sumo el valor numerico (ASCII) de sus letras. Esa suma la divido entre 100 y el residuo (modulo) me dice en que cubeta debo meter la palabra. Asi la busqueda es rapidisima.

**¿Que pasa si dos palabras distintas caen en la misma cubeta? (Colision)**
Lo resuelvo haciendo que cada cubeta sea una pequeña lista. Si la palabra ya esta en la lista, solo le sumo 1 a su contador de frecuencia. Si no esta, la agrego al final.

In [ ]:
class TablaHash:
    def __init__(self, tamano_tabla=100):
        self.capacidad = tamano_tabla
        self.buckets = []
        
        # Creo 100 listas vacias (una por cada cubeta)
        for numero in range(self.capacidad):
            self.buckets.append([])

    def _funcion_hash(self, palabra):
        # Sumo el valor de cada letra
        suma_ascii = 0
        for letra in palabra:
            suma_ascii += ord(letra)
            
        # Aseguro que el numero caiga entre 0 y 99
        return suma_ascii % self.capacidad

    def insertar(self, palabra):
        indice = self._funcion_hash(palabra)
        bucket_destino = self.buckets[indice]

        # Resolucion de colisiones: reviso si la palabra ya esta en la cubeta
        for elemento in bucket_destino:
            if elemento[0] == palabra:
                # Ya existe, solo aumento la frecuencia
                elemento[1] += 1
                return 

        # Si no existe, la guardo como nueva: ["palabra", 1]
        bucket_destino.append([palabra, 1])

    def obtener_datos(self):
        # Recolecto todo en una sola lista final
        resultado_final = []
        for bucket in self.buckets:
            for elemento in bucket:
                resultado_final.append(elemento)
        return resultado_final


## 3. Clase Ordenador (Merge Sort y Quick Sort)

Ya que tenemos las palabras sin repetir y con su conteo, necesitamos ordenarlas alfabeticamente. 

Aqui programamos dos metodos diferentes:
*   **Merge Sort:** Parte la lista a la mitad una y otra vez hasta tener listas de 1 elemento, y luego las vuelve a juntar pero ya en orden. Su complejidad siempre es O(n log n).
*   **Quick Sort:** Elige una palabra al centro (pivote) y manda a la izquierda las que van antes en el abecedario, y a la derecha las que van despues. Luego repite esto en cada lado.

In [ ]:
class Ordenador:
    # --- MERGE SORT ---
    def merge_sort(self, lista):
        if len(lista) <= 1:
            return lista

        medio = len(lista) // 2
        izquierda = lista[:medio]
        derecha = lista[medio:]

        izquierda_ordenada = self.merge_sort(izquierda)
        derecha_ordenada = self.merge_sort(derecha)

        return self._mezclar(izquierda_ordenada, derecha_ordenada)

    def _mezclar(self, izquierda, derecha):
        resultado = []
        i = 0
        j = 0

        while i < len(izquierda) and j < len(derecha):
            if izquierda[i][0] <= derecha[j][0]:
                resultado.append(izquierda[i])
                i += 1
            else:
                resultado.append(derecha[j])
                j += 1

        resultado.extend(izquierda[i:])
        resultado.extend(derecha[j:])
        return resultado

    # --- QUICK SORT ---
    def quick_sort(self, lista):
        if len(lista) <= 1:
            return lista

        pivote = lista[len(lista) // 2]
        menores = []
        iguales = []
        mayores = []

        for elemento in lista:
            if elemento[0] < pivote[0]:
                menores.append(elemento)
            elif elemento[0] == pivote[0]:
                iguales.append(elemento)
            else:
                mayores.append(elemento)

        return self.quick_sort(menores) + iguales + self.quick_sort(mayores)


## 4. Clase Cronometro (Pruebas de Tiempo)

Para cumplir con el ultimo punto de la tarea, esta clase genera textos falsos de 100, 1000 y 10000 palabras para poner a competir a mis dos algoritmos de ordenamiento y ver cual es mas rapido en la vida real.

In [ ]:
class Cronometro:
    def generar_texto(self, cantidad):
        palabras = ["manzana", "codigo", "python", "clase", "tarea", "escuela", "libro"]
        texto = []
        for i in range(cantidad):
            texto.append(palabras[i % len(palabras)])
        return " ".join(texto)

    def ejecutar_pruebas(self):
        tamanos = [100, 1000, 10000]
        print("\n--- PRUEBAS DE TIEMPO EMPIRICO ---")
        
        normalizador = Normalizador()
        ordenador = Ordenador()

        for cantidad in tamanos:
            texto = self.generar_texto(cantidad)
            palabras = normalizador.limpiar_texto(texto)

            tabla = TablaHash(tamano_tabla=1000)
            for p in palabras:
                tabla.insertar(p)
            lista_unicas = tabla.obtener_datos()

            inicio_m = time.time()
            ordenador.merge_sort(lista_unicas)
            fin_m = time.time()

            inicio_q = time.time()
            ordenador.quick_sort(lista_unicas)
            fin_q = time.time()

            print(f"Texto de {cantidad} palabras:")
            print(f"  Merge Sort: {fin_m - inicio_m:.6f} seg")
            print(f"  Quick Sort: {fin_q - inicio_q:.6f} seg")
            print("-" * 30)


## 5. Ejecucion Principal y Analisis de Complejidad

Aqui es donde cobro vida a todas las clases anteriores. 

**El Analisis de Complejidad es el siguiente:**
*   **Tabla Hash (Busqueda e Insercion):** Su mejor caso y caso promedio es O(1) porque el calculo matematico nos lleva directo a la cubeta. El peor caso seria O(n) si todas las palabras del mundo sumaran lo mismo y cayeran en la misma cubeta (se volveria una sola lista gigante).
*   **Merge Sort:** Complejidad siempre O(n log n). Es super estable porque matematicamente siempre parte todo a la mitad.
*   **Quick Sort:** Su mejor caso es O(n log n) cuando el pivote cae bien centrado. El peor caso seria O(n^2) si por mala suerte el pivote siempre es la palabra mas pequeña del grupo.

In [ ]:
if __name__ == "__main__":
    # Hacemos una prueba pequeña para demostrar que funciona
    texto_ejemplo = "Pepe crea su diccionario con python. ¡Python es genial para Pepe!"
    print("Texto original:", texto_ejemplo)

    try:
        # 1. Limpiamos
        normalizador = Normalizador()
        palabras_sueltas = normalizador.limpiar_texto(texto_ejemplo)

        # 2. Quitamos repetidos y contamos
        tabla_hash = TablaHash(tamano_tabla=50)
        for palabra in palabras_sueltas:
            tabla_hash.insertar(palabra)
        
        datos_listos = tabla_hash.obtener_datos()

        # 3. Ordenamos alfabeticamente
        ordenador = Ordenador()
        datos_ordenados = ordenador.merge_sort(datos_listos)

        print("\nDiccionario Final Ordenado:")
        for elemento in datos_ordenados:
            print(f" -> '{elemento[0]}' aparece {elemento[1]} veces")

    except ValueError as e:
        print(e)

    # 4. Corremos las pruebas de velocidad
    pruebas = Cronometro()
    pruebas.ejecutar_pruebas()
